In [1]:
import os
import io
import copy
import random
from tqdm.auto import tqdm 

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
# Classes
CAT_CLASS = 3
DOG_CLASS = 5

# Transform
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.247, 0.243, 0.261])
])

# Load CIFAR10
train_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=True,
                                      download=True, transform=transform)
test_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=False,
                                     download=True, transform=transform)


def filter_cat_vs_dog(dataset, cat_class=CAT_CLASS, dog_class=DOG_CLASS):
    indices = [i for i, label in enumerate(dataset.targets) if label in [cat_class, dog_class]]
    new_targets = [1 if dataset.targets[i] == cat_class else 0 for i in indices]
    subset = Subset(dataset, indices)
    subset.targets = new_targets
    return subset


class CatDogDataset(Dataset):
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, _ = self.subset[idx]
        label = self.subset.targets[idx]
        return img, label


# Prepare loaders
train_loader = DataLoader(CatDogDataset(filter_cat_vs_dog(train_dataset_full)), batch_size=64, shuffle=True)
test_loader = DataLoader(CatDogDataset(filter_cat_vs_dog(test_dataset_full)), batch_size=64, shuffle=False)

print("Train/Test dataset ready.")


Files already downloaded and verified
Files already downloaded and verified
Train/Test dataset ready.


In [3]:
def load_model(path="model.pth"):
    print("\nLoading pretrained ResNet18 model...")
    model = resnet18(weights=ResNet18_Weights.DEFAULT)

    # Modify final layer for 2 classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 2)

    # Freeze all layers except layer3, layer4, fc
    for param in model.parameters():
        param.requires_grad = False
    for layer in [model.layer3, model.layer4, model.fc]:
        for param in layer.parameters():
            param.requires_grad = True

    # Load fine-tuned weights if available
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"Loaded fine-tuned weights from '{path}'")
    else:
        print(f"Warning: '{path}' not found. Model is not fine-tuned yet.")

    return model.to(device)


model = load_model()
model.eval()



Loading pretrained ResNet18 model...


/tmp/ipykernel_64761/3570508965.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path, map_location=device))


Loaded fine-tuned weights from 'model.pth'


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
class Encoder(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.features = nn.Sequential(
            trained_model.conv1,
            trained_model.bn1,
            trained_model.relu,
            trained_model.maxpool,
            trained_model.layer1,
            trained_model.layer2,
            trained_model.layer3
        )

    def forward(self, x):
        with torch.no_grad():
            return self.features(x)


encoder = Encoder(model).to(device)
encoder.eval()


Encoder(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True

## Simple Decoders

### Simple Tanh Decoder

In [5]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.decode = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.decode(x)


In [6]:
# Directory with trained decoders
save_dir = "hyperparam_simple_alpha_tanh_decoders"
os.makedirs(save_dir, exist_ok=True)

criterion_ce = nn.CrossEntropyLoss()

# Convert normalized tensor back to uint8 image
def denormalize(img_tensor):
    img = img_tensor.detach().cpu().numpy()
    img = np.clip(((img * 0.5) + 0.5) * 255, 0, 255).astype(np.uint8)
    img = np.transpose(img, (1, 2, 0))
    return img

# Save tensor as JPEG
def save_tensor_as_jpeg(img_tensor, path, quality=95):
    img = Image.fromarray(denormalize(img_tensor))
    img.save(path, "JPEG", quality=quality)

# Evaluate original test images
def evaluate_original(model, dataloader):
    total, correct, running_ce = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Baseline Evaluation", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)
    acc = correct / total
    mean_ce = running_ce / total
    return acc, mean_ce

# Evaluate reconstructions and save all JPEGs
def evaluate_recon(model, encoder, decoder, dataloader):
    total, correct, running_ce = 0, 0, 0.0
    tmp_dir = "tmp_recon_images"
    os.makedirs(tmp_dir, exist_ok=True)
    file_sizes = []

    with torch.no_grad():
        img_index = 0
        for images, labels in tqdm(dataloader, desc="Reconstruction Evaluation", leave=False):
            images, labels = images.to(device), labels.to(device)
            latent = encoder(images)
            recon = decoder(latent)
            logits = model(recon)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

            for img in recon:
                img_path = os.path.join(tmp_dir, f"recon_{img_index}.jpeg")
                save_tensor_as_jpeg(img, img_path)
                file_sizes.append(os.path.getsize(img_path))
                img_index += 1

    acc = correct / total
    mean_ce = running_ce / total
    avg_size_kb = np.mean(file_sizes) / 1024 if file_sizes else 0
    return acc, mean_ce, avg_size_kb

# Compress single image tensor to match target size
def compress_to_target_size(img_tensor, target_bytes, step=5, min_q=5, max_q=95):
    img_np = denormalize(img_tensor)
    pil_img = Image.fromarray(img_np)
    q_low, q_high = min_q, max_q
    best_img = pil_img
    while q_low <= q_high:
        q_mid = (q_low + q_high) // 2
        buf = io.BytesIO()
        pil_img.save(buf, format='JPEG', quality=q_mid)
        size = buf.getbuffer().nbytes
        if abs(size - target_bytes) <= step:
            best_img = Image.open(buf).convert("RGB")
            break
        elif size > target_bytes:
            q_high = q_mid - 1
        else:
            q_low = q_mid + 1
            best_img = Image.open(buf).convert("RGB")
    return transforms.ToTensor()(best_img)

# Evaluate original images compressed to recon avg size
def evaluate_size_matched_lossy(model, dataloader, target_kb):
    total, correct, running_ce = 0, 0, 0.0
    target_bytes = int(target_kb * 1024)
    tmp_dir = "tmp_compressed_images"
    os.makedirs(tmp_dir, exist_ok=True)

    with torch.no_grad():
        img_index = 0
        for images, labels in tqdm(dataloader, desc=f"Size-Matched Lossy ({target_kb:.1f} KB)", leave=False):
            images, labels = images.to(device), labels.to(device)
            compressed_batch = []
            for img in images:
                comp = compress_to_target_size(img, target_bytes)
                compressed_batch.append(comp)
                # Save for reference and file size consistency check
                save_tensor_as_jpeg(comp, os.path.join(tmp_dir, f"compressed_{img_index}.jpeg"))
                img_index += 1
            compressed_batch = torch.stack(compressed_batch).to(device)
            logits = model(compressed_batch)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

    acc = correct / total
    mean_ce = running_ce / total
    return acc, mean_ce

# Baseline metrics
base_acc, base_ce = evaluate_original(model, test_loader)
print(f"Baseline - Accuracy: {base_acc*100:.2f}% | Baseline CE: {base_ce:.6f}")

# Save all original test images
tmp_dir = "tmp_original_images"
os.makedirs(tmp_dir, exist_ok=True)
orig_sizes = []
for i, (img, _) in enumerate(tqdm(test_loader.dataset, desc="Saving Original Images")):
    path = os.path.join(tmp_dir, f"orig_{i}.jpeg")
    save_tensor_as_jpeg(img, path)
    orig_sizes.append(os.path.getsize(path))
orig_avg_kb = np.mean(orig_sizes) / 1024
print(f"Original avg file size: {orig_avg_kb:.2f} KB")

# Evaluate each decoder
eval_csv_path = os.path.join(save_dir, "decoder_eval_summary.csv")
eval_df = pd.DataFrame(columns=[
    "alpha", "baseline_acc", "baseline_CE",
    "recon_acc", "recon_CE", "orig_avg_KB", "recon_avg_KB",
    "compression_pct", "size_matched_target_KB",
    "size_matched_lossy_acc", "size_matched_lossy_CE"
])

decoder_files = sorted([f for f in os.listdir(save_dir) if f.endswith(".pth")])

for dec_file in decoder_files:
    alpha_str = dec_file.replace("decoder", "").replace(".pth", "")
    try:
        alpha_val = float(alpha_str)
    except ValueError:
        print(f"Skipping {dec_file} (non-numeric alpha).")
        continue

    print(f"Evaluating decoder alpha={alpha_val:.2f}")
    decoder_curr = Decoder().to(device)
    decoder_curr.load_state_dict(torch.load(os.path.join(save_dir, dec_file), map_location=device))
    decoder_curr.eval()

    recon_acc, recon_ce, recon_avg_kb = evaluate_recon(model, encoder, decoder_curr, test_loader)

    # Compute compression percentage (negative means inflation)
    compression_pct = ((orig_avg_kb - recon_avg_kb) / orig_avg_kb) * 100 if orig_avg_kb > 0 else 0

    # Determine which size to match: recon size (normal) or original size (if inflated)
    if recon_avg_kb > orig_avg_kb:
        print(f"Alpha {alpha_val:.2f}: Recon images are larger than originals "
              f"({recon_avg_kb:.2f} KB > {orig_avg_kb:.2f} KB). "
              "Comparing against original-size lossy compression instead.")
        size_ref_kb = orig_avg_kb
    else:
        size_ref_kb = recon_avg_kb

    size_matched_acc, size_matched_ce = evaluate_size_matched_lossy(model, test_loader, size_ref_kb)

    eval_df = pd.concat([eval_df, pd.DataFrame([{
        "alpha": alpha_val,
        "baseline_acc": base_acc,
        "baseline_CE": base_ce,
        "recon_acc": recon_acc,
        "recon_CE": recon_ce,
        "orig_avg_KB": orig_avg_kb,
        "recon_avg_KB": recon_avg_kb,
        "compression_pct": compression_pct,
        "size_matched_target_KB": size_ref_kb,
        "size_matched_lossy_acc": size_matched_acc,
        "size_matched_lossy_CE": size_matched_ce
    }])], ignore_index=True)

    print(f"Alpha {alpha_val:.2f}: Recon Acc={recon_acc*100:.2f}% | Recon CE={recon_ce:.6f} | "
          f"Recon Size={recon_avg_kb:.2f} KB | Compression={compression_pct:+.2f}% | "
          f"Size-Matched Target={size_ref_kb:.2f} KB | "
          f"Lossy Acc={size_matched_acc*100:.2f}% | CE={size_matched_ce:.6f}")

# Save CSV
eval_df.to_csv(eval_csv_path, index=False)
print(f"Evaluation summary saved at {eval_csv_path}")


Baseline Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

Baseline - Accuracy: 81.15% | Baseline CE: 1.527898


Saving Original Images:   0%|          | 0/2000 [00:00<?, ?it/s]

Original avg file size: 1.45 KB
Evaluating decoder alpha=0.00


/tmp/ipykernel_64761/789671783.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  decoder_curr.load_state_dict(torch.load(os.path.join(save_dir, dec_file), map_location=d

Reconstruction Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

... (299 outputs left out)

Alpha 0.99: Recon Acc=78.60% | Recon CE=3.258503 | Recon Size=1.25 KB | Compression=+13.60% | Size-Matched Target=1.25 KB | Lossy Acc=63.75% | CE=3.464314
Evaluating decoder alpha=1.00


Reconstruction Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

Size-Matched Lossy (1.2 KB):   0%|          | 0/32 [00:00<?, ?it/s]

Alpha 1.00: Recon Acc=79.05% | Recon CE=3.398978 | Recon Size=1.22 KB | Compression=+16.09% | Size-Matched Target=1.22 KB | Lossy Acc=63.85% | CE=3.485475
Evaluation summary saved at hyperparam_simple_alpha_tanh_decoders/decoder_eval_summary.csv


### Simple Sigmoid Decoder

In [7]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.decode = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decode(x)


In [8]:
# Directory with trained decoders
save_dir = "hyperparam_simple_alpha_sigmoid_decoders"
os.makedirs(save_dir, exist_ok=True)

criterion_ce = nn.CrossEntropyLoss()

# Convert normalized tensor back to uint8 image
def denormalize(img_tensor):
    img = img_tensor.detach().cpu().numpy()
    img = np.clip(((img * 0.5) + 0.5) * 255, 0, 255).astype(np.uint8)
    img = np.transpose(img, (1, 2, 0))
    return img

# Save tensor as JPEG
def save_tensor_as_jpeg(img_tensor, path, quality=95):
    img = Image.fromarray(denormalize(img_tensor))
    img.save(path, "JPEG", quality=quality)

# Evaluate original test images
def evaluate_original(model, dataloader):
    total, correct, running_ce = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Baseline Evaluation", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)
    acc = correct / total
    mean_ce = running_ce / total
    return acc, mean_ce

# Evaluate reconstructions and save all JPEGs
def evaluate_recon(model, encoder, decoder, dataloader):
    total, correct, running_ce = 0, 0, 0.0
    tmp_dir = "tmp_recon_images"
    os.makedirs(tmp_dir, exist_ok=True)
    file_sizes = []

    with torch.no_grad():
        img_index = 0
        for images, labels in tqdm(dataloader, desc="Reconstruction Evaluation", leave=False):
            images, labels = images.to(device), labels.to(device)
            latent = encoder(images)
            recon = decoder(latent)
            recon = recon * 2 - 1
            logits = model(recon)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

            for img in recon:
                img_path = os.path.join(tmp_dir, f"recon_{img_index}.jpeg")
                save_tensor_as_jpeg(img, img_path)
                file_sizes.append(os.path.getsize(img_path))
                img_index += 1

    acc = correct / total
    mean_ce = running_ce / total
    avg_size_kb = np.mean(file_sizes) / 1024 if file_sizes else 0
    return acc, mean_ce, avg_size_kb

# Compress single image tensor to match target size
def compress_to_target_size(img_tensor, target_bytes, step=5, min_q=5, max_q=95):
    img_np = denormalize(img_tensor)
    pil_img = Image.fromarray(img_np)
    q_low, q_high = min_q, max_q
    best_img = pil_img
    while q_low <= q_high:
        q_mid = (q_low + q_high) // 2
        buf = io.BytesIO()
        pil_img.save(buf, format='JPEG', quality=q_mid)
        size = buf.getbuffer().nbytes
        if abs(size - target_bytes) <= step:
            best_img = Image.open(buf).convert("RGB")
            break
        elif size > target_bytes:
            q_high = q_mid - 1
        else:
            q_low = q_mid + 1
            best_img = Image.open(buf).convert("RGB")
    return transforms.ToTensor()(best_img)

# Evaluate original images compressed to recon avg size
def evaluate_size_matched_lossy(model, dataloader, target_kb):
    total, correct, running_ce = 0, 0, 0.0
    target_bytes = int(target_kb * 1024)
    tmp_dir = "tmp_compressed_images"
    os.makedirs(tmp_dir, exist_ok=True)

    with torch.no_grad():
        img_index = 0
        for images, labels in tqdm(dataloader, desc=f"Size-Matched Lossy ({target_kb:.1f} KB)", leave=False):
            images, labels = images.to(device), labels.to(device)
            compressed_batch = []
            for img in images:
                comp = compress_to_target_size(img, target_bytes)
                compressed_batch.append(comp)
                # Save for reference and file size consistency check
                save_tensor_as_jpeg(comp, os.path.join(tmp_dir, f"compressed_{img_index}.jpeg"))
                img_index += 1
            compressed_batch = torch.stack(compressed_batch).to(device)
            logits = model(compressed_batch)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

    acc = correct / total
    mean_ce = running_ce / total
    return acc, mean_ce

# Baseline metrics
base_acc, base_ce = evaluate_original(model, test_loader)
print(f"Baseline - Accuracy: {base_acc*100:.2f}% | Baseline CE: {base_ce:.6f}")

# Save all original test images
tmp_dir = "tmp_original_images"
os.makedirs(tmp_dir, exist_ok=True)
orig_sizes = []
for i, (img, _) in enumerate(tqdm(test_loader.dataset, desc="Saving Original Images")):
    path = os.path.join(tmp_dir, f"orig_{i}.jpeg")
    save_tensor_as_jpeg(img, path)
    orig_sizes.append(os.path.getsize(path))
orig_avg_kb = np.mean(orig_sizes) / 1024
print(f"Original avg file size: {orig_avg_kb:.2f} KB")

# Evaluate each decoder
eval_csv_path = os.path.join(save_dir, "decoder_eval_summary.csv")
eval_df = pd.DataFrame(columns=[
    "alpha", "baseline_acc", "baseline_CE",
    "recon_acc", "recon_CE", "orig_avg_KB", "recon_avg_KB",
    "compression_pct", "size_matched_target_KB",
    "size_matched_lossy_acc", "size_matched_lossy_CE"
])

decoder_files = sorted([f for f in os.listdir(save_dir) if f.endswith(".pth")])

for dec_file in decoder_files:
    alpha_str = dec_file.replace("decoder", "").replace(".pth", "")
    try:
        alpha_val = float(alpha_str)
    except ValueError:
        print(f"Skipping {dec_file} (non-numeric alpha).")
        continue

    print(f"Evaluating decoder alpha={alpha_val:.2f}")
    decoder_curr = Decoder().to(device)
    decoder_curr.load_state_dict(torch.load(os.path.join(save_dir, dec_file), map_location=device))
    decoder_curr.eval()

    recon_acc, recon_ce, recon_avg_kb = evaluate_recon(model, encoder, decoder_curr, test_loader)

    # Compute compression percentage (negative means inflation)
    compression_pct = ((orig_avg_kb - recon_avg_kb) / orig_avg_kb) * 100 if orig_avg_kb > 0 else 0

    # Determine which size to match: recon size (normal) or original size (if inflated)
    if recon_avg_kb > orig_avg_kb:
        print(f"Alpha {alpha_val:.2f}: Recon images are larger than originals "
              f"({recon_avg_kb:.2f} KB > {orig_avg_kb:.2f} KB). "
              "Comparing against original-size lossy compression instead.")
        size_ref_kb = orig_avg_kb
    else:
        size_ref_kb = recon_avg_kb

    size_matched_acc, size_matched_ce = evaluate_size_matched_lossy(model, test_loader, size_ref_kb)

    eval_df = pd.concat([eval_df, pd.DataFrame([{
        "alpha": alpha_val,
        "baseline_acc": base_acc,
        "baseline_CE": base_ce,
        "recon_acc": recon_acc,
        "recon_CE": recon_ce,
        "orig_avg_KB": orig_avg_kb,
        "recon_avg_KB": recon_avg_kb,
        "compression_pct": compression_pct,
        "size_matched_target_KB": size_ref_kb,
        "size_matched_lossy_acc": size_matched_acc,
        "size_matched_lossy_CE": size_matched_ce
    }])], ignore_index=True)

    print(f"Alpha {alpha_val:.2f}: Recon Acc={recon_acc*100:.2f}% | Recon CE={recon_ce:.6f} | "
          f"Recon Size={recon_avg_kb:.2f} KB | Compression={compression_pct:+.2f}% | "
          f"Size-Matched Target={size_ref_kb:.2f} KB | "
          f"Lossy Acc={size_matched_acc*100:.2f}% | CE={size_matched_ce:.6f}")

# Save CSV
eval_df.to_csv(eval_csv_path, index=False)
print(f"Evaluation summary saved at {eval_csv_path}")


Baseline Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

Baseline - Accuracy: 81.15% | Baseline CE: 1.527898


Saving Original Images:   0%|          | 0/2000 [00:00<?, ?it/s]

Original avg file size: 1.45 KB
Evaluating decoder alpha=0.00


/tmp/ipykernel_20491/613882449.py:154: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  decoder_curr.load_state_dict(torch.load(os.path.join(save_dir, dec_file), map_location=d

Reconstruction Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

... (299 outputs left out)

Alpha 0.99: Recon Acc=78.85% | Recon CE=3.129433 | Recon Size=1.14 KB | Compression=+21.29% | Size-Matched Target=1.14 KB | Lossy Acc=63.65% | CE=3.545818
Evaluating decoder alpha=1.00


Reconstruction Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

Size-Matched Lossy (1.1 KB):   0%|          | 0/32 [00:00<?, ?it/s]

Alpha 1.00: Recon Acc=79.05% | Recon CE=3.050426 | Recon Size=1.13 KB | Compression=+22.45% | Size-Matched Target=1.13 KB | Lossy Acc=63.75% | CE=3.546681
Evaluation summary saved at hyperparam_simple_alpha_sigmoid_decoders/decoder_eval_summary.csv


## Mirrored ResNet Decoder

In [7]:
class MirrorBasicBlock(nn.Module):
    """Reverse of a ResNet BasicBlock, preserving residual skip."""
    def __init__(self, block):
        super().__init__()
        # Mirror convs
        self.deconv2 = self.mirror_conv(block.conv2)
        self.bn2 = nn.BatchNorm2d(block.conv2.in_channels)
        self.deconv1 = self.mirror_conv(block.conv1)
        self.bn1 = nn.BatchNorm2d(block.conv1.in_channels)
        self.relu = nn.ReLU(inplace=True)

        # Mirror downsample (if present)
        if block.downsample is not None:
            down = block.downsample[0]
            self.upsample = nn.Sequential(
                self.mirror_conv(down),
                nn.BatchNorm2d(down.in_channels)
            )
        else:
            self.upsample = None

    def mirror_conv(self, conv):
        """Create a ConvTranspose2d layer that mirrors a given Conv2d layer."""
        return nn.ConvTranspose2d(
            in_channels=conv.out_channels,
            out_channels=conv.in_channels,
            kernel_size=conv.kernel_size,
            stride=conv.stride,
            padding=conv.padding,
            output_padding=1 if conv.stride[0] > 1 else 0,
            bias=False,
        )

    def forward(self, x):
        identity = x
        out = self.deconv2(x)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.deconv1(out)
        out = self.bn1(out)

        if self.upsample is not None:
            identity = self.upsample(identity)

        out += identity
        out = self.relu(out)
        return out


### Mirrored ResNet Tanh Decoder

In [8]:
class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()

        base = models.resnet18(weights=None)

        def mirror_layer(layer):
            blocks = [MirrorBasicBlock(b) for b in reversed(layer)]
            return nn.Sequential(*blocks)

        self.up3 = mirror_layer(base.layer3)
        self.up2 = mirror_layer(base.layer2)
        self.up1 = mirror_layer(base.layer1)

        # Mirror of initial conv (7x7 stride=2)
        self.final = nn.ConvTranspose2d(
            64, 3, kernel_size=7, stride=2, padding=3, output_padding=1
        )

        # Extra step: bring from 16×16 -> 32×32
        self.upsample_final = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.up3(x)
        x = self.up2(x)
        x = self.up1(x)
        x = self.final(x)
        x = self.upsample_final(x)
        return x

decoder = Decoder().to(device)
model = model.to(device)

In [9]:
# Directory with trained decoders
save_dir = "hyperparam_MR_alpha_tanh_decoders"
os.makedirs(save_dir, exist_ok=True)

criterion_ce = nn.CrossEntropyLoss()

# Convert normalized tensor back to uint8 image
def denormalize(img_tensor):
    img = img_tensor.detach().cpu().numpy()
    img = np.clip(((img * 0.5) + 0.5) * 255, 0, 255).astype(np.uint8)
    img = np.transpose(img, (1, 2, 0))
    return img

# Save tensor as JPEG
def save_tensor_as_jpeg(img_tensor, path, quality=95):
    img = Image.fromarray(denormalize(img_tensor))
    img.save(path, "JPEG", quality=quality)

# Evaluate original test images
def evaluate_original(model, dataloader):
    total, correct, running_ce = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Baseline Evaluation", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)
    acc = correct / total
    mean_ce = running_ce / total
    return acc, mean_ce

# Evaluate reconstructions and save all JPEGs
def evaluate_recon(model, encoder, decoder, dataloader):
    total, correct, running_ce = 0, 0, 0.0
    tmp_dir = "tmp_recon_images"
    os.makedirs(tmp_dir, exist_ok=True)
    file_sizes = []

    with torch.no_grad():
        img_index = 0
        for images, labels in tqdm(dataloader, desc="Reconstruction Evaluation", leave=False):
            images, labels = images.to(device), labels.to(device)
            latent = encoder(images)
            recon = decoder(latent)
            logits = model(recon)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

            for img in recon:
                img_path = os.path.join(tmp_dir, f"recon_{img_index}.jpeg")
                save_tensor_as_jpeg(img, img_path)
                file_sizes.append(os.path.getsize(img_path))
                img_index += 1

    acc = correct / total
    mean_ce = running_ce / total
    avg_size_kb = np.mean(file_sizes) / 1024 if file_sizes else 0
    return acc, mean_ce, avg_size_kb

# Compress single image tensor to match target size
def compress_to_target_size(img_tensor, target_bytes, step=5, min_q=5, max_q=95):
    img_np = denormalize(img_tensor)
    pil_img = Image.fromarray(img_np)
    q_low, q_high = min_q, max_q
    best_img = pil_img
    while q_low <= q_high:
        q_mid = (q_low + q_high) // 2
        buf = io.BytesIO()
        pil_img.save(buf, format='JPEG', quality=q_mid)
        size = buf.getbuffer().nbytes
        if abs(size - target_bytes) <= step:
            best_img = Image.open(buf).convert("RGB")
            break
        elif size > target_bytes:
            q_high = q_mid - 1
        else:
            q_low = q_mid + 1
            best_img = Image.open(buf).convert("RGB")
    return transforms.ToTensor()(best_img)

# Evaluate original images compressed to recon avg size
def evaluate_size_matched_lossy(model, dataloader, target_kb):
    total, correct, running_ce = 0, 0, 0.0
    target_bytes = int(target_kb * 1024)
    tmp_dir = "tmp_compressed_images"
    os.makedirs(tmp_dir, exist_ok=True)

    with torch.no_grad():
        img_index = 0
        for images, labels in tqdm(dataloader, desc=f"Size-Matched Lossy ({target_kb:.1f} KB)", leave=False):
            images, labels = images.to(device), labels.to(device)
            compressed_batch = []
            for img in images:
                comp = compress_to_target_size(img, target_bytes)
                compressed_batch.append(comp)
                # Save for reference and file size consistency check
                save_tensor_as_jpeg(comp, os.path.join(tmp_dir, f"compressed_{img_index}.jpeg"))
                img_index += 1
            compressed_batch = torch.stack(compressed_batch).to(device)
            logits = model(compressed_batch)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

    acc = correct / total
    mean_ce = running_ce / total
    return acc, mean_ce

# Baseline metrics
base_acc, base_ce = evaluate_original(model, test_loader)
print(f"Baseline - Accuracy: {base_acc*100:.2f}% | Baseline CE: {base_ce:.6f}")

# Save all original test images
tmp_dir = "tmp_original_images"
os.makedirs(tmp_dir, exist_ok=True)
orig_sizes = []
for i, (img, _) in enumerate(tqdm(test_loader.dataset, desc="Saving Original Images")):
    path = os.path.join(tmp_dir, f"orig_{i}.jpeg")
    save_tensor_as_jpeg(img, path)
    orig_sizes.append(os.path.getsize(path))
orig_avg_kb = np.mean(orig_sizes) / 1024
print(f"Original avg file size: {orig_avg_kb:.2f} KB")

# Evaluate each decoder
eval_csv_path = os.path.join(save_dir, "decoder_eval_summary.csv")
eval_df = pd.DataFrame(columns=[
    "alpha", "baseline_acc", "baseline_CE",
    "recon_acc", "recon_CE", "orig_avg_KB", "recon_avg_KB",
    "compression_pct", "size_matched_target_KB",
    "size_matched_lossy_acc", "size_matched_lossy_CE"
])

decoder_files = sorted([f for f in os.listdir(save_dir) if f.endswith(".pth")])

for dec_file in decoder_files:
    alpha_str = dec_file.replace("decoder", "").replace(".pth", "")
    try:
        alpha_val = float(alpha_str)
    except ValueError:
        print(f"Skipping {dec_file} (non-numeric alpha).")
        continue

    print(f"Evaluating decoder alpha={alpha_val:.2f}")
    decoder_curr = Decoder().to(device)
    decoder_curr.load_state_dict(torch.load(os.path.join(save_dir, dec_file), map_location=device))
    decoder_curr.eval()

    recon_acc, recon_ce, recon_avg_kb = evaluate_recon(model, encoder, decoder_curr, test_loader)

    # Compute compression percentage (negative means inflation)
    compression_pct = ((orig_avg_kb - recon_avg_kb) / orig_avg_kb) * 100 if orig_avg_kb > 0 else 0

    # Determine which size to match: recon size (normal) or original size (if inflated)
    if recon_avg_kb > orig_avg_kb:
        print(f"Alpha {alpha_val:.2f}: Recon images are larger than originals "
              f"({recon_avg_kb:.2f} KB > {orig_avg_kb:.2f} KB). "
              "Comparing against original-size lossy compression instead.")
        size_ref_kb = orig_avg_kb
    else:
        size_ref_kb = recon_avg_kb

    size_matched_acc, size_matched_ce = evaluate_size_matched_lossy(model, test_loader, size_ref_kb)

    eval_df = pd.concat([eval_df, pd.DataFrame([{
        "alpha": alpha_val,
        "baseline_acc": base_acc,
        "baseline_CE": base_ce,
        "recon_acc": recon_acc,
        "recon_CE": recon_ce,
        "orig_avg_KB": orig_avg_kb,
        "recon_avg_KB": recon_avg_kb,
        "compression_pct": compression_pct,
        "size_matched_target_KB": size_ref_kb,
        "size_matched_lossy_acc": size_matched_acc,
        "size_matched_lossy_CE": size_matched_ce
    }])], ignore_index=True)

    print(f"Alpha {alpha_val:.2f}: Recon Acc={recon_acc*100:.2f}% | Recon CE={recon_ce:.6f} | "
          f"Recon Size={recon_avg_kb:.2f} KB | Compression={compression_pct:+.2f}% | "
          f"Size-Matched Target={size_ref_kb:.2f} KB | "
          f"Lossy Acc={size_matched_acc*100:.2f}% | CE={size_matched_ce:.6f}")

# Save CSV
eval_df.to_csv(eval_csv_path, index=False)
print(f"Evaluation summary saved at {eval_csv_path}")


Baseline Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

Baseline - Accuracy: 81.15% | Baseline CE: 1.527898


Saving Original Images:   0%|          | 0/2000 [00:00<?, ?it/s]

Original avg file size: 1.45 KB
Evaluating decoder alpha=0.00


/tmp/ipykernel_64761/324736463.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  decoder_curr.load_state_dict(torch.load(os.path.join(save_dir, dec_file), map_location=d

Reconstruction Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

... (328 outputs left out)

Alpha 0.99: Recon Acc=79.15% | Recon CE=2.976210 | Recon Size=1.31 KB | Compression=+9.81% | Size-Matched Target=1.31 KB | Lossy Acc=63.90% | CE=3.450804
Evaluating decoder alpha=1.00


Reconstruction Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

Size-Matched Lossy (1.3 KB):   0%|          | 0/32 [00:00<?, ?it/s]

Alpha 1.00: Recon Acc=79.05% | Recon CE=3.163099 | Recon Size=1.30 KB | Compression=+10.20% | Size-Matched Target=1.30 KB | Lossy Acc=63.95% | CE=3.454218
Evaluation summary saved at hyperparam_MR_alpha_tanh_decoders/decoder_eval_summary.csv


### Mirrored ResNet Sigmoid Decoder

In [12]:
class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()

        base = models.resnet18(weights=None)

        def mirror_layer(layer):
            blocks = [MirrorBasicBlock(b) for b in reversed(layer)]
            return nn.Sequential(*blocks)

        self.up3 = mirror_layer(base.layer3)
        self.up2 = mirror_layer(base.layer2)
        self.up1 = mirror_layer(base.layer1)

        # Mirror of initial conv (7x7 stride=2)
        self.final = nn.ConvTranspose2d(
            64, 3, kernel_size=7, stride=2, padding=3, output_padding=1
        )

        # Extra step: bring from 16×16 -> 32×32
        self.upsample_final = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.up3(x)
        x = self.up2(x)
        x = self.up1(x)
        x = self.final(x)
        x = self.upsample_final(x)
        return x

decoder = Decoder().to(device)
model = model.to(device)

In [13]:
# Directory with trained decoders
save_dir = "hyperparam_MR_alpha_sigmoid_decoders"
os.makedirs(save_dir, exist_ok=True)

criterion_ce = nn.CrossEntropyLoss()

# Convert normalized tensor back to uint8 image
def denormalize(img_tensor):
    img = img_tensor.detach().cpu().numpy()
    img = np.clip(((img * 0.5) + 0.5) * 255, 0, 255).astype(np.uint8)
    img = np.transpose(img, (1, 2, 0))
    return img

# Save tensor as JPEG
def save_tensor_as_jpeg(img_tensor, path, quality=95):
    img = Image.fromarray(denormalize(img_tensor))
    img.save(path, "JPEG", quality=quality)

# Evaluate original test images
def evaluate_original(model, dataloader):
    total, correct, running_ce = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Baseline Evaluation", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)
    acc = correct / total
    mean_ce = running_ce / total
    return acc, mean_ce

# Evaluate reconstructions and save all JPEGs
def evaluate_recon(model, encoder, decoder, dataloader):
    total, correct, running_ce = 0, 0, 0.0
    tmp_dir = "tmp_recon_images"
    os.makedirs(tmp_dir, exist_ok=True)
    file_sizes = []

    with torch.no_grad():
        img_index = 0
        for images, labels in tqdm(dataloader, desc="Reconstruction Evaluation", leave=False):
            images, labels = images.to(device), labels.to(device)
            latent = encoder(images)
            recon = decoder(latent)
            recon = recon * 2 - 1
            logits = model(recon)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

            for img in recon:
                img_path = os.path.join(tmp_dir, f"recon_{img_index}.jpeg")
                save_tensor_as_jpeg(img, img_path)
                file_sizes.append(os.path.getsize(img_path))
                img_index += 1

    acc = correct / total
    mean_ce = running_ce / total
    avg_size_kb = np.mean(file_sizes) / 1024 if file_sizes else 0
    return acc, mean_ce, avg_size_kb

# Compress single image tensor to match target size
def compress_to_target_size(img_tensor, target_bytes, step=5, min_q=5, max_q=95):
    img_np = denormalize(img_tensor)
    pil_img = Image.fromarray(img_np)
    q_low, q_high = min_q, max_q
    best_img = pil_img
    while q_low <= q_high:
        q_mid = (q_low + q_high) // 2
        buf = io.BytesIO()
        pil_img.save(buf, format='JPEG', quality=q_mid)
        size = buf.getbuffer().nbytes
        if abs(size - target_bytes) <= step:
            best_img = Image.open(buf).convert("RGB")
            break
        elif size > target_bytes:
            q_high = q_mid - 1
        else:
            q_low = q_mid + 1
            best_img = Image.open(buf).convert("RGB")
    return transforms.ToTensor()(best_img)

# Evaluate original images compressed to recon avg size
def evaluate_size_matched_lossy(model, dataloader, target_kb):
    total, correct, running_ce = 0, 0, 0.0
    target_bytes = int(target_kb * 1024)
    tmp_dir = "tmp_compressed_images"
    os.makedirs(tmp_dir, exist_ok=True)

    with torch.no_grad():
        img_index = 0
        for images, labels in tqdm(dataloader, desc=f"Size-Matched Lossy ({target_kb:.1f} KB)", leave=False):
            images, labels = images.to(device), labels.to(device)
            compressed_batch = []
            for img in images:
                comp = compress_to_target_size(img, target_bytes)
                compressed_batch.append(comp)
                # Save for reference and file size consistency check
                save_tensor_as_jpeg(comp, os.path.join(tmp_dir, f"compressed_{img_index}.jpeg"))
                img_index += 1
            compressed_batch = torch.stack(compressed_batch).to(device)
            logits = model(compressed_batch)
            loss = criterion_ce(logits, labels)
            running_ce += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

    acc = correct / total
    mean_ce = running_ce / total
    return acc, mean_ce

# Baseline metrics
base_acc, base_ce = evaluate_original(model, test_loader)
print(f"Baseline - Accuracy: {base_acc*100:.2f}% | Baseline CE: {base_ce:.6f}")

# Save all original test images
tmp_dir = "tmp_original_images"
os.makedirs(tmp_dir, exist_ok=True)
orig_sizes = []
for i, (img, _) in enumerate(tqdm(test_loader.dataset, desc="Saving Original Images")):
    path = os.path.join(tmp_dir, f"orig_{i}.jpeg")
    save_tensor_as_jpeg(img, path)
    orig_sizes.append(os.path.getsize(path))
orig_avg_kb = np.mean(orig_sizes) / 1024
print(f"Original avg file size: {orig_avg_kb:.2f} KB")

# Evaluate each decoder
eval_csv_path = os.path.join(save_dir, "decoder_eval_summary.csv")
eval_df = pd.DataFrame(columns=[
    "alpha", "baseline_acc", "baseline_CE",
    "recon_acc", "recon_CE", "orig_avg_KB", "recon_avg_KB",
    "compression_pct", "size_matched_target_KB",
    "size_matched_lossy_acc", "size_matched_lossy_CE"
])

decoder_files = sorted([f for f in os.listdir(save_dir) if f.endswith(".pth")])

for dec_file in decoder_files:
    alpha_str = dec_file.replace("decoder", "").replace(".pth", "")
    try:
        alpha_val = float(alpha_str)
    except ValueError:
        print(f"Skipping {dec_file} (non-numeric alpha).")
        continue

    print(f"Evaluating decoder alpha={alpha_val:.2f}")
    decoder_curr = Decoder().to(device)
    decoder_curr.load_state_dict(torch.load(os.path.join(save_dir, dec_file), map_location=device))
    decoder_curr.eval()

    recon_acc, recon_ce, recon_avg_kb = evaluate_recon(model, encoder, decoder_curr, test_loader)

    # Compute compression percentage (negative means inflation)
    compression_pct = ((orig_avg_kb - recon_avg_kb) / orig_avg_kb) * 100 if orig_avg_kb > 0 else 0

    # Determine which size to match: recon size (normal) or original size (if inflated)
    if recon_avg_kb > orig_avg_kb:
        print(f"Alpha {alpha_val:.2f}: Recon images are larger than originals "
              f"({recon_avg_kb:.2f} KB > {orig_avg_kb:.2f} KB). "
              "Comparing against original-size lossy compression instead.")
        size_ref_kb = orig_avg_kb
    else:
        size_ref_kb = recon_avg_kb

    size_matched_acc, size_matched_ce = evaluate_size_matched_lossy(model, test_loader, size_ref_kb)

    eval_df = pd.concat([eval_df, pd.DataFrame([{
        "alpha": alpha_val,
        "baseline_acc": base_acc,
        "baseline_CE": base_ce,
        "recon_acc": recon_acc,
        "recon_CE": recon_ce,
        "orig_avg_KB": orig_avg_kb,
        "recon_avg_KB": recon_avg_kb,
        "compression_pct": compression_pct,
        "size_matched_target_KB": size_ref_kb,
        "size_matched_lossy_acc": size_matched_acc,
        "size_matched_lossy_CE": size_matched_ce
    }])], ignore_index=True)

    print(f"Alpha {alpha_val:.2f}: Recon Acc={recon_acc*100:.2f}% | Recon CE={recon_ce:.6f} | "
          f"Recon Size={recon_avg_kb:.2f} KB | Compression={compression_pct:+.2f}% | "
          f"Size-Matched Target={size_ref_kb:.2f} KB | "
          f"Lossy Acc={size_matched_acc*100:.2f}% | CE={size_matched_ce:.6f}")

# Save CSV
eval_df.to_csv(eval_csv_path, index=False)
print(f"Evaluation summary saved at {eval_csv_path}")


Baseline Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

Baseline - Accuracy: 81.15% | Baseline CE: 1.527898


Saving Original Images:   0%|          | 0/2000 [00:00<?, ?it/s]

Original avg file size: 1.45 KB
Evaluating decoder alpha=0.00


/tmp/ipykernel_20491/171738354.py:154: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  decoder_curr.load_state_dict(torch.load(os.path.join(save_dir, dec_file), map_location=d

Reconstruction Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

... (301 outputs left out)

Alpha 0.99: Recon Acc=79.50% | Recon CE=2.452373 | Recon Size=1.24 KB | Compression=+14.68% | Size-Matched Target=1.24 KB | Lossy Acc=63.80% | CE=3.478367
Evaluating decoder alpha=1.00


Reconstruction Evaluation:   0%|          | 0/32 [00:00<?, ?it/s]

Size-Matched Lossy (1.3 KB):   0%|          | 0/32 [00:00<?, ?it/s]

Alpha 1.00: Recon Acc=78.95% | Recon CE=3.246219 | Recon Size=1.30 KB | Compression=+10.16% | Size-Matched Target=1.30 KB | Lossy Acc=63.95% | CE=3.454218
Evaluation summary saved at hyperparam_MR_alpha_sigmoid_decoders/decoder_eval_summary.csv
